# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Melina Lazzaro**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [59]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs.aima import PriorityQueue

In [79]:
# Busqueda A*
def search_algorithm_h(number_disks=5) -> (NodeHanoi, dict):

    list_disks = [i for i in range(5, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    def heuristic_function(state):
        score = 0
        # Restamos 1 por cada disco en la misma posicion que en el estado objetivo.
        for rod_index, rod in enumerate(state.rods):
            goal_rod = goal_state.rods[rod_index]
            for pos, disk in enumerate(rod):
                if pos < len(goal_rod) and disk == goal_rod[pos]:
                    score -= 1
        return score

    def priority_function(node):
        return node.state.accumulated_cost + heuristic_function(node.state)

    frontier = PriorityQueue(order='min', f=priority_function)
    frontier.append(NodeHanoi(problem.initial))
    explored = set() # Conjunto de estados ya visitados
    node_explored = 0
    max_depth = 0

    while len(frontier) != 0:
        _, node = frontier.pop()
        node_explored += 1

        explored.add(node.state)
        if node.depth > max_depth:
            max_depth = node.depth

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": max_depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics

        # Agregamos a la frontera los nodos sucesores que no hayan sido visitados
        for next_node in node.expand(problem):
            if next_node.state not in explored:
                frontier.append(next_node)

    metrics = {
        "solution_found": False,
        "nodes_explored": node_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": None,
    }
    return None, metrics

In [ ]:
# A* search algorithm
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):
    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    right_moves = {(0, 2), (2, 1), (1, 0)}
    left_moves = {(0, 1), (1, 2), (2, 0)}

    def expected_direction(disk_weight):
        if number_disks % 2 == 0:
            return "left" if disk_weight % 2 == 0 else "right"
        return "right" if disk_weight % 2 == 0 else "left"

    def expected_disk_for_move(move_number):
        k = 1
        while move_number % 2 == 0:
            move_number //= 2
            k += 1
        return k

    def nearest_expected_phase_distance(disk_weight, move_number):
        # Distancia del movimiento actual al movimiento esperado mas cercano para ese disco.
        start = 2 ** (disk_weight - 1)
        period = 2 ** disk_weight

        if move_number <= start:
            return start - move_number

        phase = (move_number - start) % period
        return min(phase, period - phase)

    # Heuristic function from the article "Heuristic function in an algorithm of First-Best search
    # for the problem of Tower of Hanoi: optimal route for n disks"
    # http://www.ptolomeo.unam.mx:8080/jspui/bitstream/132.248.52.100/15146/3/Art%C3%ADculo.pdf
    # Adaptada para tener mayor resolucion y menos empates en f(n).
    def heuristic_function(node):
        if node.action is None:
            return 0

        moved_disk = node.action.disk
        move_direction = (node.action.rod_input, node.action.rod_out)
        move_number = node.depth

        if move_direction in right_moves:
            actual_direction = "right"
        elif move_direction in left_moves:
            actual_direction = "left"
        else:
            actual_direction = None

        # Direccion: penalizacion suave si coincide, mas alta y ponderada por disco si no coincide.
        direction_penalty = 0 if actual_direction == expected_direction(moved_disk) else (1 + moved_disk)

        # Timing: penalizacion por distancia al movimiento esperado mas cercano de ese disco.
        timing_penalty = nearest_expected_phase_distance(moved_disk, move_number)

        # Desempate por cercania al objetivo: cuenta discos fuera de la posicion final correcta.
        goal_mismatch = 0
        for rod_index, rod in enumerate(node.state.rods):
            goal_rod = goal_state.rods[rod_index]
            for pos, disk in enumerate(rod):
                if pos >= len(goal_rod) or disk != goal_rod[pos]:
                    goal_mismatch += 1

        return direction_penalty + timing_penalty + goal_mismatch

    def priority_function(node):
        return node.state.accumulated_cost + heuristic_function(node)

    frontier = PriorityQueue(order='min', f=priority_function)
    frontier.append(NodeHanoi(problem.initial))
    explored = set()
    node_explored = 0
    max_depth = 0

    while len(frontier) != 0:
        _, node = frontier.pop()
        node_explored += 1

        explored.add(node.state)
        if node.depth > max_depth:
            max_depth = node.depth

        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": node_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": max_depth,
                "cost_total": node.state.accumulated_cost,
            }
            return node, metrics

        for next_node in node.expand(problem):
            if next_node.state not in explored:
                frontier.append(next_node)

    metrics = {
        "solution_found": False,
        "nodes_explored": node_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": None,
    }
    return None, metrics

Se prueba la función:

In [83]:
solution, metrics = search_algorithm_h(number_disks=5)

Veamos las métricas:

In [81]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 268
states_visited: 169
nodes_in_frontier: 18
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [68]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [69]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
